# CV13 — Theo dõi điểm qua video

Lấy query feature tại tọa độ frame đầu rồi correlate với các frame. Tọa độ chuẩn hóa [-1,1], thứ tự x,y, align_corners=False; pixel center i được đổi bằng 2*(i+0.5)/size-1.

Năm cell code, chạy lần lượt. Fixture không phải benchmark; xem REFERENCE.md để chọn dữ liệu thật và paper gốc.


## Cell 1 — Môi trường và seed

Dùng CPU để kiểm tra trước. Tìm repo root từ thư mục notebook, đặt seed và import PyTorch. Không tải pretrained weights.


In [ ]:
from pathlib import Path
import sys
import torch
from torch import nn
import torch.nn.functional as F

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'vision/curriculum').is_dir())
sys.path.insert(0, str(ROOT))
torch.manual_seed(42)
torch.set_num_threads(2)
device = torch.device('cpu')  # Change to cuda after the CPU example works.
print('Device:', device)


## Cell 2 — Tạo batch và model

Lấy query feature tại tọa độ frame đầu rồi correlate với các frame. Tọa độ chuẩn hóa [-1,1], thứ tự x,y, align_corners=False; pixel center i được đổi bằng 2*(i+0.5)/size-1. Dữ liệu trong cell là fixture tổng hợp để bạn đọc được shape; thay bằng Dataset thật trước khi so metric.


In [ ]:
from vision.curriculum.cv13_tracking import Tracker, loss_fn
frame = torch.rand(2, 1, 3, 32, 32, device=device)
video = frame.repeat(1, 4, 1, 1, 1)
points = torch.tensor([[[-0.5, -0.5], [0.0, 0.0], [0.5, 0.5]]], device=device).repeat(2, 1, 1)
trajectory = points[:, None].expand(-1, 4, -1, -1).clone()
valid = torch.ones(2, 4, 3, device=device)
model = Tracker().to(device)
batch = (video, points, trajectory, valid)


## Cell 3 — Forward và baseline

Đọc shape/output trước khi train. Các số in ra chỉ minh họa phép tính và luồng dự đoán.


In [ ]:
model.eval()
with torch.no_grad():
    prediction = model(video, points)
print('Trajectory B x T x N x XY:', prediction.shape)
print('Stationary baseline loss:', (trajectory - points[:, None]).abs().mean().item())


## Cell 4 — Một bước cập nhật

Tính đúng loss của nhiệm vụ, backward và cập nhật tham số. Một bước không phải quy trình huấn luyện hoàn chỉnh. TENT tự thực hiện update bên trong adapt.


In [ ]:
model.train()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=3e-4)
optimizer.zero_grad(set_to_none=True)
loss = loss_fn(model, *batch)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
print('One-step training loss:', loss.detach().item())


## Cell 5 — Chẩn đoán và hướng nâng cấp

Stationary → feature correlation + soft coordinates → local iterative update và visibility head. Fixture frame lặp chỉ kiểm tra đường chạy, không đo tracking chuyển động.


In [ ]:
model.eval()
with torch.no_grad():
    prediction = model(video, points)
    normalized_l1 = ((prediction - trajectory).abs().sum(-1) * valid).sum() / valid.sum()
print('Fixture normalized coordinate L1:', normalized_l1.item())
print('No visibility head yet; this is not a full TAP-Vid average-Jaccard evaluator.')


## Thử nghiệm tiếp theo

Stationary → feature correlation + soft coordinates → local iterative update và visibility head. Fixture frame lặp chỉ kiểm tra đường chạy, không đo tracking chuyển động.

Giữ nguyên split và ngân sách huấn luyện. Đọc [module](../cv13_tracking.py) và [tài liệu gốc](../REFERENCE.md).
